In [1]:
from icecream import ic
import torch
import torch.nn as nn
from datasets_exp import OxfordPetDataset
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from models import Model, DinoWrapper, ModelPeft
from transformers import get_cosine_schedule_with_warmup

In [2]:
train_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="trainval")
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=128,      # How many samples per batch
    shuffle=True,
    num_workers=8)

test_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="test")
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,      # How many samples per batch
    shuffle=False)

3680it [00:00, 2706002.58it/s]
ic| len(self.samples): 3680
3669it [00:00, 1677264.46it/s]
ic| len(self.samples): 3669


In [3]:
print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")

images, labels = next(iter(train_dataloader))
print(f"Image shape: {images.shape}")       # should be (128, 3, 448, 448)
print(f"Labels: {labels[:10]}")             # spot check
print(f"Label min/max: {labels.min()}, {labels.max()}")  # should be 0-36
print(f"Label dtype: {labels.dtype}")       # should be torch.int64
print(f"Unique classes: {labels.unique().numel()}")      # ideally close to 37 in one batch

Train size: 3680
Test size:  3669
Image shape: torch.Size([128, 3, 224, 224])
Labels: tensor([ 7, 31, 15, 20, 12, 32, 17, 11, 34,  8])
Label min/max: 0, 36
Label dtype: torch.int64
Unique classes: 37


In [4]:
target_modules = ["qkv", "proj", "fc1", "fc2"]
model = ModelPeft(num_classes=37, target_modules=target_modules)
model.cuda()

/home/system/miniconda3/envs/j/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/system/miniconda3/envs/j/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


trainable params: 6,291,456 || all params: 310,303,744 || trainable%: 2.0275


ModelPeft(
  (base): InternVisionModel(
    (embeddings): InternVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): InternVisionEncoder(
      (layers): ModuleList(
        (0-23): 24 x InternVisionEncoderLayer(
          (attn): InternAttention(
            (qkv): lora.Linear(
              (base_layer): Linear(in_features=1024, out_features=3072, bias=True)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.05, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=1024, out_features=16, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=16, out_features=3072, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (attn_drop)

In [ ]:
LR = 1e-4  # or even 3e-4
WEIGHT_DECAY = 0.01
EPOCHS = 10
WARMUP_RATIO = 0.05

device = "cuda"

# model = qwen(num_classes=37)
# model = model.to(device)
# model = DinoWrapper(num_classes=37)
# model = model.to(device)

# model = model.to(torch.bfloat16)  # if you need the classifier in bf16

# optimizer = torch.optim.AdamW(
#     model.classifier.parameters(), 
    
#     lr=LR,
#     weight_decay=WEIGHT_DECAY
# )
optimizer = torch.optim.AdamW([
        {"params": model.model_peft.parameters(), "lr": 1e-4},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ],
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

    
num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()


total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable params: {total}')
for epoch in tqdm(range(EPOCHS)):

    # ---------------- TRAIN ----------------
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for images, labels in train_dataloader:

        images = images.to(device).to(torch.bfloat16)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)           # (B, num_classes)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)

    # ---------------- VALID ----------------
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in test_dataloader:

            images = images.to(device).to(torch.bfloat16)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )

Total trainable params: 6835237


  0%|          | 0/10 [00:00<?, ?it/s]

/home/system/miniconda3/envs/j/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch [1/10] Loss: 3.7930 Train Acc: 5.46% Val Acc: 24.15%
Epoch [2/10] Loss: 1.4037 Train Acc: 65.62% Val Acc: 85.53%
Epoch [3/10] Loss: 0.2290 Train Acc: 92.23% Val Acc: 90.98%
Epoch [4/10] Loss: 0.0900 Train Acc: 96.60% Val Acc: 90.84%
Epoch [5/10] Loss: 0.0435 Train Acc: 98.80% Val Acc: 93.02%
